# Finanzierung von Wohneigentum in Österreich: 25-Jahres-Vergleich

## 🎯 Ziel
Beantwortung der Frage:
**War es in den letzten 25 Jahren in Österreich einfacher oder schwerer, ein Haus zu finanzieren?**

👉 Vergleich der monatlichen Kreditrate für ein durchschnittliches Haus mit dem Medianeinkommen.

---

## 🗂️ Datenbeschaffung

1. **Kreditzinsen**
   - Quelle: [OENB - Wohnbaukredite an private Haushalte](https://www.oenb.at/Statistik/Standardisierte-Tabellen/zinssaetze-und-wechselkurse/Zinssaetze-der-Kreditinstitute/Kreditzinss-tze---Neugesch-ft.html)
   - Daten: Wohnbaukredite an private Haushalte (1996–2024)
   - Kurzbeschreibung: Die Daten basieren auf der WWU-weit harmonisierten Zinssatzstatistik der EZB.

2. **Einkommen**
   - Quelle: [Statistik Austria - Jährliche Personeneinkommen](https://www.statistik.at/statistiken/bevoelkerung-und-soziales/einkommen-und-soziale-lage/jaehrliche-personeneinkommen)
      - Daten wurden manuell aus ods datei in csv umgewandelt
   - Daten: Median des Nettojahreseinkommen der unselbständig Erwerbstätigen 1997 bis 2023 (Gesamt)
   - Kurzbeschreibung: Die jährlichen Personeneinkommen umfassen Nettojahreseinkommen von unselbständig Erwerbstätigen basierend auf sozialstatistischen Auswertungen der Lohnsteuerdaten.

3. **Immobilienpreise**
   - Quelle: - [OENB - Wohnimmobilienpreisindex](https://www.oenb.at/Statistik/Standardisierte-Tabellen/Preise-Wettbewerbsfaehigkeit/immobilien/wohnimmobilienpreisindex.html)
   - Daten: Wohnimmobilien Österreich (2000 - 2024)
   - Kurzbeschreibung: Der Index basiert auf Quadratmeterpreisen für neue und gebrauchte Eigentumswohnungen sowie Einfamilienhäuser und wird seit Q3 2017 mit einer verfeinerten Methodik berechnet, die auf der Doppelimputation und einem generalisierten additiven Modell basiert.
   - Um den Immobilienpreis vom index zu berechnen, wurde der [Preis für Wohnhäuser im Jahr 2024 (2.709 Euro/m²)](https://www.statistik.at/statistiken/volkswirtschaft-und-oeffentliche-finanzen/preise-und-preisindizes/immobilien-durchschnittspreise) verwendet.

## Hinweise zur Berechnung

- Annahmen:
    - Kreditanteil: 80% des Kaufpreises (20% Eigenkapital)
    - Laufzeit: 25 Jahre (300 Monate)
    - Hausgröße: 130 m²
    - Zinssatz: fix (da einfacher zu berechnen)
    - Monatliches Einkommen: Jährliches Nettoeinkommen durch 12 und multipliziert mit 1,5 (Haushaltseinkommen geschätzt)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Immobilienpreise

df_prices = pd.read_csv('data/OeNB_6_6_Wohnimmobilienpreisindex_20250927_160408.csv', delimiter=';', encoding='latin1')
df_prices['Werte'] = df_prices['Werte'].str.replace(',', '.').astype(float)

indicator = 'Österreich -Wohnimmobilienpreisindex 2000=100 Hedon. Reggr.-Modell'
df = df_prices[df_prices['Indikator'] == indicator]
df = df[['Jahr', 'Werte']]

index_2024 = df[df['Jahr'] == 2024]['Werte'].values[0]
price_2024 = 2709  # Euro/m² für Wohnhäuser
df['Preis_Euro_m2'] = df['Werte'] / index_2024 * price_2024

In [ ]:
# Einkommensdaten
df_income = pd.read_csv('data/netto_jahreseinkommen.csv', delimiter=';', encoding='UTF-8')

df['Netto_Jahreseinkommen_Median'] = df['Jahr'].map(df_income.set_index('Jahr')['Nettojahreseinkommen Median'])
df['Netto_Monatseinkommen_Median'] = df['Netto_Jahreseinkommen_Median'] / 12 * 1.5

In [ ]:
# Kreditzinssätze
df_interest = pd.read_csv('data/OeNB_2_10_1_KreditzinssätzeNeugeschäft_20250927_170134.csv', delimiter=';', encoding='latin1')
df_interest = df_interest.set_index(' in % p. a.').T
df_interest = df_interest.reset_index().rename(columns={'index': 'Jahr'})
df_interest['Jahr'] = df_interest['Jahr'].astype(int)
df_interest['Wohnbaukredite an private Haushalte'] = df_interest['Wohnbaukredite an private Haushalte'].str.replace(',', '.').astype(float)

df['Zinssatz'] = df['Jahr'].map(df_interest.set_index('Jahr')['Wohnbaukredite an private Haushalte'])

In [ ]:
# Berechnung aller Werte

kreditanteil = 0.8
laufzeit_jahre = 25
laufzeit_monate = laufzeit_jahre * 12
haus_groese_m2 = 130
spalten_name_preis = 'Preis_Euro_' + str(haus_groese_m2) + 'm2'

df[spalten_name_preis] = df['Preis_Euro_m2'] * haus_groese_m2
df['Kreditbetrag'] = df[spalten_name_preis] * kreditanteil
df['Kreditbetrag_mit_zinsen'] = df['Kreditbetrag'] * ((df['Zinssatz'] / 100) / (1 - (1 + (df['Zinssatz'] / 100))**(-laufzeit_jahre))) * laufzeit_jahre
# df['Kreditbetrag_mit_zinsen'] = df['Kreditbetrag'] * (1 + df['Zinssatz']/100)**laufzeit_jahre
df['Monatliche_Rate'] = df['Kreditbetrag_mit_zinsen'] / (laufzeit_jahre * 12)
df['Monatliche_Rate_in_prozent_des_einkommen'] = df['Monatliche_Rate'] / (df['Netto_Monatseinkommen_Median']) * 100

jahre_beispiele = [2000, 2005, 2008, 2010, 2020, 2023]  # Liste der relevanten Jahre

for jahr_beispiel in jahre_beispiele:
    hauspreis_beispiel = df[df['Jahr'] == jahr_beispiel][spalten_name_preis].values[0]
    kreditbetrag_beispiel = df[df['Jahr'] == jahr_beispiel]['Kreditbetrag'].values[0]
    zinssatz_beispiel = df[df['Jahr'] == jahr_beispiel]['Zinssatz'].values[0]
    kreditbetrag_mit_zinsen_beispiel = df[df['Jahr'] == jahr_beispiel]['Kreditbetrag_mit_zinsen'].values[0]
    monatliche_rate_beispiel = df[df['Jahr'] == jahr_beispiel]['Monatliche_Rate'].values[0]
    monatliches_netto_einkommen = df[df['Jahr'] == jahr_beispiel]['Netto_Monatseinkommen_Median'].values[0]
    anteil_am_einkommen = df[df['Jahr'] == jahr_beispiel]['Monatliche_Rate_in_prozent_des_einkommen'].values[0]

    print(f"Beispiel Jahr {jahr_beispiel}:")
    print(f"  - Hauspreis: {hauspreis_beispiel:.2f} €")
    print(f"  - Kreditbetrag (80%): {kreditbetrag_beispiel:.2f} €")
    print(f"  - Zinssatz: {zinssatz_beispiel:.2f} %")
    print(f"  - Kreditbetrag mit Zinsen (25 Jahre): {kreditbetrag_mit_zinsen_beispiel:.2f} €")
    print(f"  - Monatliche Rate: {monatliche_rate_beispiel:.2f} €")
    print(f"  - Monatliches Netto-Einkommen: {monatliches_netto_einkommen:.2f} €")
    print(f"  - Anteil der Rate am Einkommen: {anteil_am_einkommen:.2f} %")



In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 8))

color = 'tab:blue'
ax1.set_xlabel('Jahr', fontsize=12)
ax1.set_ylabel('Hauspreis / Gesamtbetrag (' + str(haus_groese_m2) + 'm²) in €', fontsize=12)
ax1.plot(df['Jahr'], df[spalten_name_preis], color=color, label='Hauspreis (' + str(haus_groese_m2) + 'm²)', linewidth=2)
ax1.plot(df['Jahr'], df['Kreditbetrag_mit_zinsen'], color='tab:orange', label='Gesamtbetrag (inkl. Zinsen)', linewidth=2, linestyle='--')
ax1.tick_params(axis='y')
ax1.tick_params(axis='x', rotation=45)
ax1.legend(loc='upper center', fontsize=10)
ax1.grid(visible=True, which='major', linestyle='--', linewidth=0.5)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Zinssatz (%)', color=color, fontsize=12)
ax2.plot(df['Jahr'], df['Zinssatz'], color=color, label='Zinssatz (%)', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)
ax2.legend(loc='upper right', fontsize=10)

plt.title('Hauspreis, Zinssatz und Gesamtbetrag über die Jahre', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 8))

color = 'tab:blue'
ax1.set_xlabel('Jahr', fontsize=12)
ax1.set_ylabel('Betrag in €', fontsize=12, color=color)
ax1.plot(df['Jahr'], df['Monatliche_Rate'], color='tab:blue', label='Monatliche Rate', linewidth=2)
ax1.plot(df['Jahr'], df['Netto_Monatseinkommen_Median'], color='tab:green', label='Monatliches Netto-Einkommen', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(visible=True, which='major', linestyle='--', linewidth=0.5)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Monatliche Rate in % des Nettoeinkommens', fontsize=12, color=color)
ax2.plot(df['Jahr'], df['Monatliche_Rate_in_prozent_des_einkommen'], color=color, label='Rate in % des Einkommens', linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)
ax2.legend(loc='upper right', fontsize=10)

plt.title('Monatliche Rate, Netto-Einkommen und Prozentsatz der Rate (2000-2023)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df['Jahr'], df['Monatliche_Rate_in_prozent_des_einkommen'], marker='o', color='tab:green', linewidth=2)
ax.set_xlabel('Jahr', fontsize=12)
ax.set_ylabel('Monatliche Rate in % des Nettoeinkommens', fontsize=12)
ax.set_title('Leistbarkeit von Wohneigentum in Österreich (2000-2023)', fontsize=14, fontweight='bold')
ax.axhline(y=30, color='r', linestyle='--', label='30% Einkommensgrenze')
ax.legend()
ax.grid(visible=True, which='major', linestyle='--', linewidth=0.5)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()